In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
import joblib

print("="*80)
print("🚀 EXACT NOTEBOOK CELL 4 PKL MAKER (fixed for deployment)")
print("="*80)

df = pd.read_csv(r'C:\Users\mdsal\Downloads\Telegram Desktop\SPAS_Dataset_200000_enhanced_with_soil_rainfall_fixed.csv')

# ==================== YOUR CLEANING (Cells 3-5) ====================
df = df.dropna(subset=['Season'])
df = df[df['Area'] > 0]
df['Yield (tons/hectare)'] = df['Production'] / df['Area']
df['Yield (tons/hectare)'] = df['Yield (tons/hectare)'].clip(upper=20)
df = df[(df['Yield (tons/hectare)'] > 0.1) & (df['Yield (tons/hectare)'] < 15)]
df = df[df['Yield (tons/hectare)'] > 0]
df = df[df['Yield (tons/hectare)'] <= 200]
df = df[(df['Min Temp'] <= df['Avg Temp']) & (df['Avg Temp'] <= df['Max Temp'])]
df = df[(df['Avg Humidity'] > 0) & (df['Avg Humidity'] <= 100)]
df = df[(df['Max Relative Humidity'] > 0) & (df['Max Relative Humidity'] <= 100)]
df = df[(df['Min Relative Humidity'] > 0) & (df['Min Relative Humidity'] <= 100)]
df['Min Relative Humidity'], df['Max Relative Humidity'] = df[['Min Relative Humidity', 'Max Relative Humidity']].min(axis=1), df[['Min Relative Humidity', 'Max Relative Humidity']].max(axis=1)
df['Crop Name'] = df['Crop Name'].str.strip().str.title()
df['District'] = df['District'].str.strip().str.title()
df['Season'] = df['Season'].str.strip().str.title()
if 'Monthly_Avg_Rainfall_mm' in df.columns:
    df = df.dropna(subset=['Monthly_Avg_Rainfall_mm'])
    df = df[(df['Monthly_Avg_Rainfall_mm'] >= 0) & (df['Monthly_Avg_Rainfall_mm'] <= 1000)]
if 'Primary_Soil_Type' in df.columns:
    df = df.dropna(subset=['Primary_Soil_Type'])
    df = df[df['Primary_Soil_Type'].str.strip() != '']

print(f"FINAL CLEANED DATA: {len(df):,} rows")

# ==================== CELL 4 EXACT (fixed order) ====================
# Save original strings for dummies
season_original = df['Season'].copy()
district_original = df['District'].copy()
soil_original = df['Primary_Soil_Type'].copy()

# Label encoders for yield model
season_le = LabelEncoder()
soil_le = LabelEncoder()
df['Season'] = season_le.fit_transform(season_original)
df['Primary_Soil_Type'] = soil_le.fit_transform(soil_original)

# YIELD MODEL (9 features - NO Area)
yield_feature_names = [
    "Avg Temp", "Avg Humidity", "Max Temp", "Min Temp",
    "Max Relative Humidity", "Min Relative Humidity",
    "Monthly_Avg_Rainfall_mm", "Season", "Primary_Soil_Type"
]
X_yield = df[yield_feature_names]
y_yield = df["Yield (tons/hectare)"]

yield_model = DecisionTreeRegressor(max_depth=25, random_state=42)
yield_model.fit(X_yield, y_yield)

# CROP MODEL (one-hot)
feature_columns_crop = [
    "Avg Temp", "Avg Humidity", "Max Temp", "Min Temp",
    "Max Relative Humidity", "Min Relative Humidity"
]
X_crop_raw = df[feature_columns_crop].copy()

season_dummies = pd.get_dummies(season_original, prefix="Season", drop_first=True)
district_dummies = pd.get_dummies(district_original, prefix="District", drop_first=True)
soil_dummies = pd.get_dummies(soil_original, prefix="Soil", drop_first=True)

X_crop = X_crop_raw.join([season_dummies, district_dummies, soil_dummies])
le_crop = LabelEncoder()
y_crop = le_crop.fit_transform(df["Crop Name"])

crop_model = KNeighborsClassifier(n_neighbors=5)
crop_model.fit(X_crop, y_crop)

# ==================== SAVE EVERYTHING ====================
joblib.dump(yield_model, "yield_model.pkl")
joblib.dump(crop_model, "crop_model.pkl")
joblib.dump(le_crop, "label_encoder_crop.pkl")
joblib.dump(season_le, "label_encoder_season.pkl")
joblib.dump(soil_le, "label_encoder_soil.pkl")

joblib.dump(yield_feature_names, "yield_feature_names.pkl")
joblib.dump(X_crop.columns.tolist(), "crop_feature_names.pkl")

# Save classes for UI
joblib.dump(season_original.unique().tolist(), "season_classes.pkl")
joblib.dump(district_original.unique().tolist(), "district_classes.pkl")
joblib.dump(soil_original.unique().tolist(), "soil_classes.pkl")

print("\n🎉 ALL FILES SAVED SUCCESSFULLY!")
print("Files created:")
for f in ["yield_model.pkl", "crop_model.pkl", "label_encoder_crop.pkl", "label_encoder_season.pkl", "label_encoder_soil.pkl",
          "yield_feature_names.pkl", "crop_feature_names.pkl", "season_classes.pkl", "district_classes.pkl", "soil_classes.pkl"]:
    print(f"   • {f}")

🚀 EXACT NOTEBOOK CELL 4 PKL MAKER (fixed for deployment)
FINAL CLEANED DATA: 169,069 rows

🎉 ALL FILES SAVED SUCCESSFULLY!
Files created:
   • yield_model.pkl
   • crop_model.pkl
   • label_encoder_crop.pkl
   • label_encoder_season.pkl
   • label_encoder_soil.pkl
   • yield_feature_names.pkl
   • crop_feature_names.pkl
   • season_classes.pkl
   • district_classes.pkl
   • soil_classes.pkl
